# E0: is a dense, warped, related task worth anything to a Bayes learner predicting a scarce one?

## Description

**Question.** A Meta-PFN would predict a scarce target task A (n_A = 0–8 points) from A's points *and* a dense source task B: a previous HPO run on a related dataset, whose landscape is related to A's through an unknown, non-uniform warp of the inputs. Before building one, the information has to be there. **Under a pair prior of this kind, how much better does a Bayes learner that knows the prior, but not the pair's warp or relatedness, predict A's held-out points than A alone does? And how much of the known-warp headroom survives having to infer the warp?** If a Bayes learner gains nothing, no amortized model can (the predecessor's E1 question, never run at full size).

**Setup.**
- **Pairs** (`pairs.py`). One latent function f is drawn from pfn-repro's BNN prior: the prior the frozen PFNs were trained on, on the same ECDF scale. B observes f directly (n_B = 32 uniform points). A observes it through a per-axis forward map T from A's box into a sub-window of B's (n_A ≤ 8 nested points, 32 held-out queries), all with the PFN's training noise. The strata are:
  - `identity`: T = id;
  - `window`: per axis T(x) = a + b x with b ∈ [0.4, 1];
  - `warp`: a window composed with a Kumaraswamy CDF, log α, log β ~ N(0, 0.5²);
  - `unrelated`: B observes an independent function.

  **Relatedness σ** perturbs A's BNN weights variance-preservingly. σ = 0 is the same function; σ = 0.05 and 0.1 give median Spearman(f_A, f_B) ≈ 0.9 and 0.7, measured at d = 1, 2 in the log (§2).
- **Predictors**, all from the frozen `pfn_large` (`oracle.py`):
  - `lower` = PFN(A), A alone;
  - `naive` = PFN(B ∪ A), pooling without registration;
  - `oracle_T` = PFN(B ∪ T(A)) at T(x\*), with the warp known;
  - `bayes`: the posterior predictive of a learner whose prior is the pair prior, a mixture over {unrelated, identity, 128–256 window and 128–256 warp hypotheses}. Each hypothesis is weighted by the PFN's chain-rule likelihood of A's points given B. It treats B as exact data under each hypothesis, so at σ > 0 it is a model-averaging learner rather than the exact Bayes one: a *lower* bound on what B is worth.
- **Grid.** d ∈ {1, 2} × {identity, window, warp} × σ ∈ {0, 0.05, 0.1}, plus `unrelated`, for 20 cells. Pilot: 50 pairs per cell (pair split 0). Confirm: fresh pairs (split 1).

**What to look for.**
- **Gains over A alone**, in nats per held-out query, with 95% pair-bootstrap CIs, as a function of n_A:
  - `gain_bayes` (what B is worth);
  - `gain_oracle` (with T known);
  - `gain_naive` (without registration);
  - G = gain_bayes / gain_oracle (the share of the headroom that survives inferring T).
- **Diagnostics.**
  - the posterior mass per relation kind;
  - the particles' ESS;
  - the oracle's Monte-Carlo error: an independent hypothesis set on 10 pairs per cell.

**Decision rule (fixed before the run; `analysis/headroom.py::verdict`).**
- **INVALID** unless, on identity pairs at σ = 0 and n_A = 8, the Bayes learner recovers ≥ 90% of the oracle gain, and the oracle gain is > 0 in every σ = 0 gate cell.
- **Validity.** A gate cell whose oracle MC error exceeds 0.025 nats is imprecise. An imprecise oracle under-estimates gains, so it can block KILL but never support it.
- **KILL** (stop the Meta-PFN line on this prior family) if in every gate cell (window and warp × σ ∈ {0, 0.05, 0.1} × n_A ∈ {2, 4, 8} × d ∈ {1, 2}) the CI upper bound of gain_bayes is < 0.05 nats.
- **GO** (build and train a Meta-PFN on this prior) if, in the go cells (window and warp at **σ = 0.1**, the least related level, × n_A ∈ {2, 4} × both d), the CI lower bound of gain_bayes is > 0.10 nats **and** G > 0.25.
- **NARROW** otherwise: restrict the prior to where the gain holds, then re-run.
- *Safety (reported with the verdict).* On unrelated pairs the CI lower bound of gain_bayes should stay above −0.05, so that negative transfer is avoided by the learner itself.

**Limits.**
- Per-axis warps only, no rotations or general axis coupling. That is the realistic HPO case (per-axis shifts and stretches), and it matches the predecessor's spec v4 at L1–L3.
- Noise is the BNN prior's own. No failures and no output maps per task.
- B's design is uniform, not trace-biased.
- d ≤ 2: the particle oracle degrades with the number of warp parameters, and the ESS and MC error say where.
- The chain-rule likelihood depends on the order of A's design. It is exact only for a Bayes-optimal PFN.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meta_pfn.analysis import load_snapshot, show_header
from meta_pfn.analysis.headroom import gains, mc_error, verdict

SNAPSHOT = "results/headroom/<sweep_id>"  # set after `just snapshot`
snap = load_snapshot(SNAPSHOT)
show_header(snap)

## Results

All numbers are nats per held-out A query; gains are relative to A alone (positive = B helps). CIs: 95% pair bootstrap.

In [ ]:
cells = snap.runs.set_index("member")[["task.d", "task.kind", "task.sigma"]].set_axis(["d", "kind", "sigma"], axis=1)
def with_cell(name):
    return snap.table(name).join(cells, on="member")
nll, post = with_cell("nll"), with_cell("posterior")
g = gains(nll)
mc = mc_error(with_cell("mc"))
g.loc[g.n_a.isin([2, 4, 8])].pivot_table(index=["d", "kind", "sigma"], columns="n_a", values=["gain_bayes", "gain_oracle", "G"]).round(3)

### Gain over A alone, by relation and relatedness

Solid: Bayes learner (warp inferred); dashed: warp known; dotted: naive pooling. Shaded: 95% CI of the Bayes gain.

In [ ]:
kinds = ["identity", "window", "warp", "unrelated"]
fig, axes = plt.subplots(2, 4, figsize=(17, 7), sharex=True)
for r, d in enumerate(sorted(g.d.unique())):
    for c, kind in enumerate(kinds):
        ax = axes[r, c]
        for k, (s, h) in enumerate(g.loc[(g.d == d) & (g.kind == kind)].groupby("sigma")):
            col = f"C{k}"
            ax.plot(h.n_a, h.gain_bayes, "-o", c=col, ms=3, label=f"σ={s}")
            ax.fill_between(h.n_a, h.gain_bayes_lo, h.gain_bayes_hi, color=col, alpha=0.15)
            if kind != "unrelated":
                ax.plot(h.n_a, h.gain_oracle, "--", c=col, lw=1)
            ax.plot(h.n_a, h.gain_naive, ":", c=col, lw=1)
        ax.axhline(0, c="k", lw=0.8); ax.axhline(0.1, c="gray", lw=0.6, ls=":")
        ax.set_xscale("symlog", linthresh=1); ax.set_xticks([0, 1, 2, 4, 8], [0, 1, 2, 4, 8])
        ax.set_title(f"d = {d}, {kind}"); ax.grid(alpha=0.3)
        if c == 0: ax.set_ylabel("gain over A alone (nats/query)")
        if r == 1: ax.set_xlabel("n_A")
axes[0, 0].legend(fontsize=8); fig.tight_layout()

### Does the learner know what B is?

Posterior mass per relation kind (mean over pairs), and the oracle's precision: particle ESS and Monte-Carlo error (independent hypothesis set on 10 pairs per cell).

In [ ]:
pm = post.groupby(["d", "kind", "sigma", "n_a"])[["p_unrelated", "p_identity", "p_window", "p_warp"]].mean()
display(pm.loc[(slice(None), slice(None), slice(None), [0, 2, 8]), :].round(2))
ess = post.groupby(["d", "kind", "sigma", "n_a"])[["ess_window", "ess_warp"]].median().unstack("n_a")
display(ess.round(1))
mc.pivot_table(index=["d", "kind", "sigma"], columns="n_a", values="mc_error").round(3)

## Observations

_Filled from the outputs above._

In [ ]:
res = verdict(g, mc)
res

## Verdict

_From the rule above; recorded with `just decide`._